###** Part 1 — Python & Data Foundations: ingest, validate, clean, enrich (26 marks)**

## TASK 1.1-READ THE FILE THE FOUNDATION WAY

In [ ]:
import csv
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

folder_path = '/content/drive/MyDrive/careloop_project/data/'

print(os.listdir(folder_path))

['appointments.csv', 'followup_notes.csv', 'clinics.csv', 'clinics_enriched.csv', 'load_data.sql', 'appointments_clean.csv']


In [ ]:
with open("/content/drive/MyDrive/careloop_project/data/appointments.csv","r",newline="") as file:
  reader = csv.DictReader(file)
  rows = list(reader)
  print("Column names : ")
  print(reader.fieldnames)
  print("Total number of rows:",len(rows))

Column names : 
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total number of rows: 1020


In [ ]:
def parse_row(row):
  row["age"] = int(row["age"])
  row["fee_inr"] = int(row["fee_inr"])
  row["prior_no_shows"] = int(row["prior_no_shows"])
  row["distance_km"] = float(row["distance_km"])
  return row

In [ ]:
accepted = []
rejected = []

for row in rows:

  try:
    converted_row = parse_row(row)
    accepted.append(converted_row)

  except ValueError:
    rejected.append(row)
print("accepted rows:", len(accepted))
print("rejected rows:", len(rejected))
print("total rows:", len(accepted) + len(rejected))


accepted rows: 988
rejected rows: 32
total rows: 1020


In [ ]:
with open("rejected_rows.json","w") as file:
  json.dump(rejected,file,indent=2)

  print("rejected records saved successfully!")

rejected records saved successfully!


## TASK 1.2 - PANDAS INSPECTION AND CLEANING

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/drive/MyDrive/careloop_project/data/appointments.csv")

In [ ]:
# 1. CHECK DATASET DIMENSIONS
print("dataset shape:", df.shape)
#2. check coloumn names and data type
df.info()
#3. check missing values
print("\nMissing values:")
print(df.isnull().sum())
#4. check duplicate rows
print("\nduplicate rows:", df.duplicated().sum())

dataset shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clini

In [ ]:
#step 1 Remove exact duplicates

print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates()

print("rows after removing duplicates:", len(df))

Rows before removing duplicates: 1020
rows after removing duplicates: 1000


In [ ]:
# step 2  clean booking_channel

df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print(df["booking_channel"].value_counts())



booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64


In [ ]:
# step 3: convert date columns
df["appt_date"] = pd.to_datetime(

 df["appt_date"], errors = "coerce")

df["booked_on"] = pd.to_datetime(df["booked_on"])

print(df[["appt_date", "booked_on"]].dtypes)

appt_date    datetime64[ns]
booked_on    datetime64[ns]
dtype: object


In [ ]:
# step 4 : drop rows with missing date and statuses

print("rows before dropping missing values:",len(df))

df = df.dropna(subset=["appt_date", "status"])

print("rows after drpping missing values:", len(df))

rows before dropping missing values: 1000
rows after drpping missing values: 990


## 1.3 CLEAN, PASS2 : VALIDITY, GAPS AND OUTLIERS


In [ ]:
# STEP 1 : IDENTIFY INVALID AGES

df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Missing ages after validation:", df["age"].isnull().sum())


Missing ages after validation: 17


In [ ]:
# STEP 2: FILL MISSING VALUES  USING MEDIAN
#fill missing age with eadian

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

# fill missing distance with median
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# check missing values again

print("missing age:", df["age"].isnull().sum())

print("missing distance:", df["distance_km"].isnull().sum())




missing age: 0
missing distance: 0


In [ ]:
df = df.copy()

In [ ]:
# STEP 3: CHECK ROW COUNTS AT EACH STAGE

original = pd.read_csv(
    "/content/drive/MyDrive/careloop_project/data/appointments.csv"
)

print("Original rows:", len(original))
print("Duplicate rows:", original.duplicated().sum())

cleaned = original.drop_duplicates().copy()

print("After removing duplicates:", len(cleaned))

cleaned["appt_date"] = pd.to_datetime(
    cleaned["appt_date"], errors="coerce"
)

cleaned["booked_on"] = pd.to_datetime(
    cleaned["booked_on"], errors="coerce"
)

print("Missing appointment dates:", cleaned["appt_date"].isna().sum())
print("Missing statuses:", cleaned["status"].isna().sum())

print("Rows missing date OR status:",
      cleaned[["appt_date", "status"]].isna().any(axis=1).sum())

cleaned = cleaned.dropna(
    subset=["appt_date", "status"]
)

print("Final rows:", len(cleaned))

Original rows: 1020
Duplicate rows: 20
After removing duplicates: 1000
Missing appointment dates: 6
Missing statuses: 4
Rows missing date OR status: 10
Final rows: 990


In [ ]:
# STEP 4
# USE THE VERIFIED CLEANED DATASET
df = cleaned.copy()

# clean booking channel
df["booking_channel"] = (df["booking_channel"].str.strip().str.title())


# validate age
df.loc[(df["age"] < 1) | (df["age"] > 100),"age"] = np.nan

# fill missing age

df["age"] = df["age"].fillna(
    df["age"].median()
).round().astype(int)

# fill missing distance

df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

print("final rows:",len(df))
print("missing age:",df["age"].isnull().sum())
print("missing distance:",df["distance_km"].isnull().sum())





final rows: 990
missing age: 0
missing distance: 0


In [ ]:
# step 5 : OUTLIER DETECTION
# detect and cap outlier

q1_distance = df["distance_km"].quantile(0.25)
q3_distance = df["distance_km"].quantile(0.75)

iqr_distance = q3_distance - q1_distance

lower_distance = q1_distance - 1.5*iqr_distance

upper_distance = q3_distance + 1.5*iqr_distance

distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()

df["distance_km"] = np.clip(
    df["distance_km"],lower_distance, upper_distance
)

# fee outliers

q1_fee = df["fee_inr"].quantile(0.25)
q3_fee = df["fee_inr"].quantile(0.75)
iqr_fee = q3_fee - q1_fee

lower_fee = q1_fee - 1.5 * iqr_fee

upper_fee = q3_fee  + 1.5 * iqr_fee

fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

df["fee_inr"] = np.clip(
    df["fee_inr"], lower_fee , upper_fee
)

print("distance value capped:", distance_capped)

print("fee value capped:",fee_capped)

distance value capped: 26
fee value capped: 3


In [ ]:
# STEP 6 : FINAL VALIDATION

print("final dataset shape:", df.shape)

print("\nmissing values:")
print(df.isnull().sum())

print("\nduplicate rows:",df.duplicated().sum())

print("\ndata types:")
print(df.dtypes)

final dataset shape: (990, 14)

missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
dtype: int64

duplicate rows: 0

data types:
appointment_id             object
patient_id                 object
clinic_id                  object
booked_on          datetime64[ns]
appt_date          datetime64[ns]
age                         int64
gender                     object
distance_km               float64
fee_inr                   float64
booking_channel            object
visit_type                 object
reminder_sent              object
prior_no_shows              int64
status                     object
dtype: object


##1.4  FEATURE AND A NUMPY SUMMARY

In [ ]:
#df = pd.read_csv("/content/drive/MyDrive/careloop_project/data/appointments.csv")

In [ ]:
#step 1: convert date columns into datetime

df["appt_date"] = pd.to_datetime(df["appt_date"],errors="coerce")

df["booked_on"] = pd.to_datetime(df["booked_on"],errors="coerce")


In [ ]:
#step 2 :create new features

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# create no show column
df["no_show"] = (df["status"].str.lower() =="no-show") .astype(int)

# check new columns
print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())
("\nnew dataset shape:", df.shape)

   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1


('\nnew dataset shape:', (990, 16))

In [ ]:
# step 3 calculate average lead days

avg_lead_days = np.mean(df["lead_days"])

print("average lead days:", avg_lead_days)


average lead days: 11.819191919191919


In [ ]:
# step 4 calculate no show rate

no_show_rate = np.mean(df["no_show"]) * 100
print("No-show rate:",round(no_show_rate,2),"%")

No-show rate: 22.02 %


In [ ]:
# step 5 calculate mean of each column

x = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

column_means = np.mean(x,axis=0)

print("column means:",  column_means)

column means: [ 48.17373737   9.50093434 635.71338384  11.81919192]


In [ ]:
# step 6 : find booking with lead days greater than 14

mask = df["lead_days"] > 14

count = mask.sum()
share = mask.mean() * 100

print("Booking above 14 days :", count)

print("Share of bookings:", round(share,2),"%")

Booking above 14 days : 296
Share of bookings: 29.9 %


In [ ]:
# step 7 : calculate distance z-scores

x = df["distance_km"].to_numpy()

z =(x-np.mean(x)) / np.std(x)

outlier_count = np.sum(np.abs(z) > 2)

print("distance outliers:", outlier_count)

distance outliers: 53


## TASK 1.5 - ENRICH CLINICS USING A LIVE API

In [ ]:
# Step 1 : load clinics dataset

import pandas as pd
import requests
clinics = pd.read_csv(
    "/content/drive/MyDrive/careloop_project/data/clinics.csv"
)

print("Dataset shape:",clinics.shape)

print(clinics.head())
print(clinics.columns)

Dataset shape: (8, 6)
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Index(['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude'], dtype='object')


In [ ]:
# Step 2: Fetch current temperature for all clinics

def get_temperature(latitude, longitude):

    params = {
        "latitude": float(latitude),
        "longitude": float(longitude),
        "current": "temperature_2m",
        "timezone": "auto"
    }

    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        temperature = data["current"]["temperature_2m"]
        temperature_time = data["current"]["time"]

        if temperature is None:
            raise ValueError("Temperature is missing")

        return {
            "temp_c": float(temperature),
            "temperature_time": temperature_time,
            "api_status": "success"
        }

    except (
        requests.exceptions.RequestException,
        KeyError,
        TypeError,
        ValueError
    ) as e:

        print("API error:", e)

        return {
            "temp_c": None,
            "temperature_time": None,
            "api_status": "failed"
        }

In [ ]:
# step 3:add temperature column

clinics["temp_c"] = temp_values

print(clinics.head(8))

print("Dataset shape:", clinics.shape)



  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    19.2  
1    19.5  
2    18.6  
3    14.4  
4    14.4  
5    27.9  
6    16.6  
7    14.0  
Dataset shape: (8, 7)


In [ ]:
# step 4 : save the file

clinics.to_csv(
    "/content/drive/MyDrive/careloop_project/data/clinics_enriched.csv",
    index = False
)

print("clinics file saved successfully")


clinics file saved successfully


## TASK 1.6- EXPORT THE CLEAN TABLE AND WRITE THE SQL LOAD SCRIPT

In [ ]:
# step 1 : check appointments dataset

print("shape:", df.shape)

print("columns:", df.columns.tolist())

print("Duplicate  appointment IDs:",df["appointment_id"].duplicated().sum())

print("Missing values:")
print(df.isnull().sum())

shape: (990, 16)
columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']
Duplicate  appointment IDs: 0
Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project data folder
data_dir = Path('/content/drive/MyDrive/careloop_project/data')

# Load enriched clinics
clinics = pd.read_csv(data_dir / 'clinics_enriched.csv')

# Prepare cleaned appointments
appointments_clean = df.copy()

# Sort and format dates
appointments_clean = appointments_clean.sort_values(
    'appointment_id'
).reset_index(drop=True)

for col in ['booked_on', 'appt_date']:
    appointments_clean[col] = pd.to_datetime(
        appointments_clean[col]
    ).dt.strftime('%Y-%m-%d')

# Save cleaned appointments
appointments_clean.to_csv(
    data_dir / 'appointments_clean.csv',
    index=False
)

# SQL value formatter
def sql_value(value, column, date_cols, numeric_cols):
    if pd.isna(value):
        return 'NULL'

    if column in date_cols:
        return f"'{value}'"

    if column in numeric_cols:
        return str(value)

    return "'" + str(value).replace("'", "''") + "'"


# Generate SQL statements
lines = []

clinic_cols = [
    'clinic_id', 'clinic_name', 'city',
    'tier', 'latitude', 'longitude', 'temp_c'
]

appointment_cols = [
    'appointment_id', 'patient_id', 'clinic_id',
    'booked_on', 'appt_date', 'age', 'gender',
    'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows',
    'status', 'lead_days', 'no_show'
]

clinic_numeric = ['latitude', 'longitude', 'temp_c']

appointment_numeric = [
    'age', 'distance_km', 'fee_inr',
    'prior_no_shows', 'lead_days', 'no_show'
]

date_cols = ['booked_on', 'appt_date']

# Clinic INSERT statements first
for _, row in clinics.iterrows():
    values = [
        sql_value(row[col], col, [], clinic_numeric)
        for col in clinic_cols
    ]
    lines.append(
        "INSERT INTO clinics VALUES (" +
        ",".join(values) + ");"
    )

# Appointment INSERT statements
for _, row in appointments_clean.iterrows():
    values = [
        sql_value(row[col], col, date_cols, appointment_numeric)
        for col in appointment_cols
    ]
    lines.append(
        "INSERT INTO appointments VALUES (" +
        ",".join(values) + ");"
    )

# Save SQL file
sql_path = data_dir / 'load_data.sql'

with open(sql_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print("SQL file created successfully!")
print("Clinic INSERT statements:", len(clinics))
print("Appointment INSERT statements:", len(appointments_clean))
print("Total INSERT statements:", len(lines))
print("File location:", sql_path)

SQL file created successfully!
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998
File location: /content/drive/MyDrive/careloop_project/data/load_data.sql
